# アメダス観測CSV（風・雨）変換 & push（Colab版）

手元PC（`C:\Users\kousu\typhoon-wind-rainfall`）に保存された風・雨のCSV
（`scripts/fetch_amedas_obsdl.py` で取得したもの）を、**Google Drive経由で
Colabに渡し、台風ごとのJSONに変換してGitHubへ直接push**するノートブック
です。`git`操作は一切不要です。

## 前提
- 手元PCの `data/raw_amedas/` フォルダの中身（`<台風コード>_wind.csv` /
  `<台風コード>_rain.csv` という名前のファイル）を、**まるごとGoogle Drive
  にアップロード**しておくこと
- GitHubへpushする権限（Personal Access Token）を用意すること（②で説明）

## 使い方
①→②→③→④→⑤の順に上から実行してください。

## ① Google Driveをマウントし、CSVフォルダを指定

`data/raw_amedas/` の中身（`*_wind.csv` / `*_rain.csv`）をアップロードした
Drive上のフォルダパスを指定してください。フォルダ名は何でも構いません
（中のファイル名が `<台風コード>_wind.csv` 等になっていれば認識します）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 例: '/content/drive/MyDrive/raw_amedas'
CSV_DIR = '/content/drive/MyDrive/raw_amedas'

import pathlib
csvs = list(pathlib.Path(CSV_DIR).glob('*.csv'))
print(f'{CSV_DIR} 内に csv ファイル {len(csvs)} 個')
print('例:', [p.name for p in csvs[:5]])

## ② GitHubへのPersonal Access Token（PAT）を用意

すでに持っていれば③に進んでOKです。まだの場合:

1. GitHubの https://github.com/settings/tokens?type=beta を開く
2. **Generate new token** → このリポジトリ（`awg-yk/typhoon-wind-rainfall`）
   に対して **Contents: Read and write** 権限を付与
3. 発行されたトークン（`github_pat_...` から始まる文字列）をコピー

次のセルを実行すると入力欄が出るので、そこに貼り付けてください
（画面には表示されず、Colab上にも保存されません）。

In [ ]:
import getpass

GITHUB_TOKEN = getpass.getpass('GitHubのPersonal Access Tokenを貼り付けてEnter: ')

## ③ リポジトリを取得し、CSVを配置

In [ ]:
import os, shutil, pathlib

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/typhoon-dataset-improvements-hn814c'
REMOTE_URL = f'https://{GITHUB_TOKEN}@github.com/awg-yk/typhoon-wind-rainfall.git'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REMOTE_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!cd {REPO_DIR} && git config user.email "colab@example.com"
!cd {REPO_DIR} && git config user.name "Colab"

# CSVをリポジトリの data/raw_amedas/ にコピー（convert_amedas_csv.py が
# 参照する固定パス）
raw_amedas = pathlib.Path(REPO_DIR) / 'data' / 'raw_amedas'
raw_amedas.mkdir(parents=True, exist_ok=True)
n = 0
for p in pathlib.Path(CSV_DIR).glob('*.csv'):
    shutil.copy(p, raw_amedas / p.name)
    n += 1
print(f'{n} 個のCSVを {raw_amedas} にコピーしました')

## ④ 変換実行

`data/raw_amedas/*_{wind,rain}.csv` を台風ごとの
`data/storms_obs/<台風コード>_{wind,rain}.json` に変換します。

In [ ]:
!cd {REPO_DIR} && python3 scripts/convert_amedas_csv.py --all

import pathlib
out_dir = pathlib.Path(REPO_DIR) / 'data' / 'storms_obs'
files = list(out_dir.glob('*.json'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} JSON files, {total_mb:.1f} MB total')

## ⑤ GitHubへコミット & push

`data/storms_obs/` 配下のJSONだけをコミットします
（`git status` の出力で他のファイルが混ざっていないか一応確認してください）。

In [ ]:
!cd {REPO_DIR} && git add data/storms_obs/
!cd {REPO_DIR} && git status --short
!cd {REPO_DIR} && git commit -m "Add AMeDAS wind/rain observation data (data/storms_obs/*.json)"
!cd {REPO_DIR} && git push origin {BRANCH}

## 完了後

- GitHub上の該当ブランチに `data/storms_obs/*.json` が反映されていれば成功
  です。
- フロントエンド（`index.html`）は、その台風に `data/storms_obs/<コード>_
  {wind,rain}.json` があれば自動的にAMeDAS観測網（風976地点・雨1669地点）で
  表示します（統合済み）。特別な追加作業は不要です。
- このノートブックのセッションを閉じれば、貼り付けたトークンはColab上から
  消えます。念のため、使い終わったトークンはGitHubの設定画面から失効させて
  おくとより安全です。